# Merging frequential & LIV data for VLC

## 1 - Import of all experimental Data in miscellaneous
- Import of f-3dB vs J data stored in miscellaneous (ALEDIA measurement only)

In [ ]:
import os
import pandas as pd
import seaborn as sns

# Path of the folder to scan
Folder = r"W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measurements"

# List initiation to store the dataframes
dfs = []
df_final = []
dfs = []

# Screening of of files contained in the folder
for root, dirs, files in os.walk(Folder):
    for f in files:
        if "DeviceData.txt" in f:
            complete_path = os.path.join(root, f)
            # Reading of the file (adapt separator if needed)
            #print(complete_path)
            df_temp = pd.read_csv(complete_path, sep='\s+', header=1)
            df_temp['path']=complete_path
            dfs.append(df_temp) #precise append VS concat

# Concatenate all dataframes
if dfs:  # check that at least 1 file has been found
    df_final = pd.concat(dfs, ignore_index=True) #what does it do
else:
    print("No files DeviceData.txt found.")

df_final = df_final.dropna(subset=["f3dB"])

#Addition of Nwire theoretical and current from current density only working for PT1 so far
df_final['Nw_theoretical']= pd.to_numeric(df_final['Device'].str.split("_").str[2].str.split("-").str[0],errors='coerce')
df_final['current']= df_final['CurrentDensity']*(25*df_final['Nw_theoretical']*1e-8)

#Plot the figure if needed to check good data import
#fig = sns.scatterplot(data=df_final, x="CurrentDensity", y="f3dB", hue="Device")



<>:20: SyntaxWarning:

invalid escape sequence '\s'

<>:20: SyntaxWarning:

invalid escape sequence '\s'

C:\Users\ymalier\AppData\Local\Temp\ipykernel_22852\75774561.py:20: SyntaxWarning:

invalid escape sequence '\s'



1       W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measur...
2       W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measur...
3       W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measur...
4       W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measur...
5       W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measur...
                              ...                        
2753    W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measur...
2754    W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measur...
2755    W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measur...
2756    W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measur...
2757    W:\Ateliers\Mesure\MISCELLANEOUS\289-RF_Measur...
Name: path, Length: 2407, dtype: object

## 2 - Import LIV of all devices measured in RF from GOZER database.
- Create a unique list of devices name

In [ ]:
import psycopg2
import numpy as np

#Get the list of sample to consider from df_final
Samples_list = df_final['Device'].unique().tolist()

#Creation of separated columns wafer,X,Y,sample from Devices and store them in df_parsed
parsed = []

for e in Samples_list:
    wafer, rest = e.split("_", 1)
    xy, sample = rest.split("_", 1)
    xposition, yposition = xy.split(",")

    parsed.append((wafer, xposition, yposition, sample))

df_parsed = pd.DataFrame(parsed, columns=["wafer", "xposition", "yposition", "sample"])
print(parsed)

#Creat a list to the good format for SQL filter
Filter_Samples = ", ".join(
    f"('{w}','{x}','{y}','{s}')" 
    for w, x, y, s in parsed
)

#SQL request with filtered samples
#Latest for only last measurement in Test Date
SQL_REQUEST = f"""
WITH latest AS (
    SELECT 
        w.wafer_name,
        epi."Run_Name",
        epi_mv."Pocket",
        dev."Block_X_Coordinate", 
        dev."Block_Y_Coordinate",
        dev."Led_Name",
        test."Test_Date", 
	    fdl."Request_Number",
        UNNEST(test_l."I") AS Current,
        UNNEST(test_l."V") AS Voltage,
        UNNEST(test_l."L") AS L,
	    UNNEST(test_l."EQE") AS "EQE",
	    UNNEST(test_l."WPE") AS "WPE",
	    UNNEST(test_l."Lambda_Peak") AS "Lambda_Peak",
	    UNNEST(test_l."Lambda_Dominant") AS "Lambda_Dominant",
        ROW_NUMBER() OVER (
            PARTITION BY w.wafer_name, dev."Block_X_Coordinate", dev."Block_Y_Coordinate", dev."Led_Name"
            ORDER BY test."Test_Date" DESC
        ) AS rn
    FROM public.wafer w 
	    LEFT OUTER JOIN public."Devices" dev 
		    ON  (dev.fk_wafer = w.pk_wafer )
        LEFT OUTER JOIN public."EpitaxyMeasuredValue" epi_mv 
 		    ON  ( epi_mv.fk_wafer = w.pk_wafer ) 	
        LEFT OUTER JOIN public."TestLed" test_l 
            ON  ( test_l."FK_Device" = dev."PK_Devices" )  
        LEFT OUTER JOIN public."Epitaxie" epi 
            ON  ( epi."PK_Epitaxie" = epi_mv."FK_Epitaxy" )  
        LEFT OUTER JOIN public."Test" test 
            ON  ( test."PK_Test" = test_l."FK_Test" )
        LEFT OUTER JOIN public."FDL" fdl 
            ON  ( epi_mv.fk_fdl = fdl."PK_FDL" )
    WHERE (w.wafer_name, dev."Block_X_Coordinate", dev."Block_Y_Coordinate", dev."Led_Name") IN ({Filter_Samples}))
    SELECT *
    FROM latest
    WHERE rn = 1;
"""

#Launch of SQL request with connection to GOZER
def read_gozer_data_liv() -> pd.DataFrame:
    conn = psycopg2.connect(
            host="INF10",
            database="GOZER",
            user="*****", #ton identifiant
            port='5432',
            password="*****") #mot de passe gozer
    cursor = conn.cursor()
    cursor.execute(SQL_REQUEST)  # execute the query
    data = cursor.fetchall()  # fetch all of the rows from the query
    col_name = [desc[0] for desc in cursor.description]
    cursor.close()  # close the communication with the PostgreSQL
    conn.close()
    df=pd.DataFrame(data, columns=col_name)
    return df

df_liv=read_gozer_data_liv()

#Addition of the columns required for further calculations
df_liv["X,Y"]=df_liv["Block_X_Coordinate"].astype(str) + "," + df_liv["Block_Y_Coordinate"].astype(str)
df_liv["Device"]=df_liv["wafer_name"].astype(str) + "_" + df_liv["X,Y"].astype(str) + "_" + df_liv["Led_Name"].astype(str)
df_liv["current"] = pd.to_numeric(df_liv["current"], errors="coerce")
df_liv_clean=df_liv
df_liv_clean['Abs_current']= np.absolute(df_liv['current'])


# Printing of columns names and figuer to check data import 
# print(df_liv_clean.keys())
# fig = sns.scatterplot(data=df_liv_clean, x="current", y="voltage", hue="Device")


[('BP16', '1', '1', '18-HV2'), ('BP16', '3', '3', '2x18-HV2'), ('95HRF2T6MMB0', '1', '3', '77-D1'), ('95HRF2T6MMB0', '4', '3', '77-D1'), ('95HRF2T6MMB0', '6', '3', '77-D1'), ('95HRF2T6MMB0', '2', '4', '77-D1'), ('95HRF2T6MMB0', '4', '2', '77-D1'), ('95HRF2T6MMB0', '5', '2', '77-D1'), ('67UIO146MMA5', '0', '4', '77-D1'), ('67UIO156MMF1', '3', '3', '438-D3'), ('67UIO1I6MMG5', '6', '4', '438-D3'), ('67UIO146MMA5', '5', '3', '438-D3'), ('67UIO1G6MMF0', '5', '4', '210-D2'), ('94FWF616MMF7', '1', '3', '666-D4'), ('67UIO1I6MMG5', '3', '2', '875-D7'), ('67UIO1G6MMF0', '5', '4', '875-D7'), ('275BL6D6MMA2', '1', '7', '4-BSM41B-4W-HV1-V1-1'), ('275BL6D6MMA2', '4', '1', '4-BSM41B-4W-HV1-V1-1'), ('275BL6D6MMA2', '-3', '4', '4-BSM41B-4W-HV1-V1-1'), ('94G2F4X6MMA2', '4', '7', '210-D2'), ('94G2F4H6MMC0', '0', '5', '210-D2'), ('94G2F4H6MMC0', '2', '7', '210-D2'), ('94G2F4H6MMC0', '2', '8', '210-D2'), ('94G2F4W6MMD1', '1', '5', '210-D2'), ('94G2F4W6MMD1', '6', '5', '210-D2'), ('94G2F4W6MMD1', '6', '6', 

## 2 - Figure mixing LIV & RF data
## a- Interpolation of LIV at current values for which f-3dB is measured

In [43]:
import numpy as np
import pandas as pd

keys = ["Device"]

df_liv2 = df_liv_clean.copy()
df_liv2["type"] = "liv"

df_freq2 = df_final.copy()
df_freq2["type"] = "freq"

cols_liv = ["current", "l", "EQE", "WPE"]
cols_freq = ["current", "f3dB"]

for col in cols_liv:
    df_liv2[col] = pd.to_numeric(df_liv2[col], errors="coerce")

for col in cols_freq:
    df_freq2[col] = pd.to_numeric(df_freq2[col], errors="coerce")

df_liv2 = df_liv2.dropna(subset=cols_liv)
df_freq2 = df_freq2.dropna(subset=cols_freq)

df_all = pd.concat([df_liv2, df_freq2], ignore_index=True)

def interpolate_device(group):
    liv = group[group["type"] == "liv"].sort_values("current")
    freq = group[group["type"] == "freq"].sort_values("current")

    #If you want to check data for every device in liv & freq tables
    #dev = group["Device"].iloc[0]   # ← récupérer le nom du device
    #print(f"{dev} → liv={len(liv)}, freq={len(freq)}")

    if liv.empty or freq.empty:
        return pd.DataFrame()   # Never none

    # Linear interpolation
    L_interp = np.interp(freq["current"], liv["current"], liv["l"])
    V_interp = np.interp(freq["current"], liv["current"], liv["voltage"])
    Lambdap_interp = np.interp(freq["current"], liv["current"], liv["Lambda_Peak"])    
    Lambdad_interp = np.interp(freq["current"], liv["current"], liv["Lambda_Dominant"])
    EQE_interp = np.interp(freq["current"], liv["current"], liv["EQE"])
    WPE_interp = np.interp(freq["current"], liv["current"], liv["WPE"])

    # Additions of the interpolated values in new columns of a copy of frequency table
    out = freq.copy()
    out["L_interp"] = L_interp
    out["V_interp"] = V_interp
    out["Lambdap_interp"] = Lambdap_interp
    out["Lambdad_interp"] = Lambdad_interp
    out["EQE_interp"] = EQE_interp
    out["WPE_interp"] = WPE_interp

    return out

#Groupby columns together
df_interp = (
    df_all
    .groupby("Device", group_keys=False)
    .apply(interpolate_device)
    .reset_index(drop=True) #à comprendre et commenter
)

df_interp

C:\Users\ymalier\AppData\Local\Temp\ipykernel_22852\3357511595.py:60: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,wafer_name,Run_Name,Pocket,Block_X_Coordinate,Block_Y_Coordinate,Led_Name,Test_Date,Request_Number,current,voltage,...,Magnitude_Normalized_Smoothed,f6dB,Ratio_f6dB_f3dB,Nw_theoretical,L_interp,V_interp,Lambdap_interp,Lambdad_interp,EQE_interp,WPE_interp
0,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.001050,NaN,...,"1.16958067615,1.27435841023,1.28436090628,1.23...",1.353250e+09,2.775328,210.0,0.000007,2.862980,464.241270,467.125060,0.152490,1.422656
1,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.002625,NaN,...,"1.00754307741,1.14602370792,1.28728876877,1.39...",1.363200e+09,2.174509,210.0,0.000141,3.176518,459.713262,463.098126,1.899268,16.002388
2,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.005250,NaN,...,"0.480223774008,0.559850400297,0.634598536653,0...",1.412950e+09,1.479142,210.0,0.000571,3.585672,455.282618,460.025009,3.961923,30.072699
3,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.010500,NaN,...,"0.33715734751,0.373869796149,0.4181271208,0.44...",1.552250e+09,1.147053,210.0,0.001207,4.010669,451.902971,458.280569,4.194975,28.718070
4,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.026250,NaN,...,"0.19887214104,0.210849295076,0.274311968864,0....",1.761200e+09,1.113204,210.0,0.002388,4.713104,448.855619,457.559275,3.310647,19.442139
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1772,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.000963,NaN,...,NaN,NaN,NaN,77.0,0.000145,5.260958,449.409147,455.801032,5.686446,30.039167
1773,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.001925,NaN,...,NaN,NaN,NaN,77.0,0.000232,5.814174,447.774820,455.221996,4.521825,21.632678
1774,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.003850,NaN,...,NaN,NaN,NaN,77.0,0.000350,6.371722,446.250385,454.938973,3.296919,14.389919
1775,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,0.009625,NaN,...,NaN,NaN,NaN,77.0,0.000579,7.127366,444.846640,455.261454,2.239172,8.781707


## b- Plot of the following graphs in a Dash interface
- JV
- EQE VS J 
- f-3dB VS J
- lambda VS J
- f-3dB VS EQE
- f-3dB VS WPE
- f-3dB VS L(W)
- f-3dB VS Voltage(V)

In [ ]:
from dash import Dash, dcc, html, Input, Output, dash_table
import plotly.express as px


app = Dash(__name__)

app.layout = html.Div([
    html.H2("Select Device"),

    dcc.Dropdown(
        id="device-select",
        options=[{"label": d, "value": d} for d in sorted(df_interp["Device"].unique())],
        value=sorted(df_interp["Device"].unique())[0],
        clearable=False
    ),
    html.Div([
        dcc.Graph(id="graph-current-v"),
        dcc.Graph(id="graph-eqe-current"),
        dcc.Graph(id="graph-lambda-current"),
        dcc.Graph(id="graph-f3db-current"),
    ], style={"display": "flex", "flexDirection": "row"}),
    html.Div([
        dcc.Graph(id="graph-f3db-l"),
        dcc.Graph(id="graph-f3db-eqe"),
        dcc.Graph(id="graph-f3db-wpe"),        
        dcc.Graph(id="graph-f3db-v"),
    ], style={"display": "flex", "flexDirection": "row"}),
])

@app.callback(
    Output("graph-current-v", "figure"),
    Output("graph-eqe-current", "figure"),
    Output("graph-lambda-current", "figure"),
    Output("graph-f3db-current", "figure"),
    Output("graph-f3db-eqe", "figure"),
    Output("graph-f3db-wpe", "figure"),
    Output("graph-f3db-l", "figure"),
    Output("graph-f3db-v", "figure"),
    Input("device-select", "value")
)

def update_graphs(device):

    df_graph = df_interp[df_interp["Device"] == device].sort_values("CurrentDensity")
    df_liv_final=df_liv_clean[df_liv_clean["Device"]==device]

    # 1) CurrentDensity vs V
    fig1 = px.line(
        df_liv_final,
        x="voltage",
        y="Abs_current",
        title=f"{device} — Current vs V"
    )
    fig1.update_traces(mode="lines+markers")
    fig1.update_yaxes(type="log")
    fig1.update_layout(
        xaxis_title="Voltage (V)",
        yaxis_title="Current density (A/cm²)"
    )
 

    # 2) EQE vs CurrentDensity
    fig2 = px.line(
        df_graph,
        x="CurrentDensity",
        y="EQE_interp",
        title=f"{device} — EQE vs Current density"
    )
    fig2.update_traces(mode="lines+markers")
    fig2.update_xaxes(type="log")
    fig2.update_layout(
        xaxis_title="Current density (A/cm²)",
        yaxis_title="EQE (%)"
    )

    # 3) Lambda vs CurrentDensity
    fig3 = px.line(
        df_graph,
        x="CurrentDensity",
        y="Lambdap_interp",
        title=f"{device} — λp vs Current density"
    )
    fig3.update_traces(mode="lines+markers")
    fig3.update_xaxes(type="log")
    fig3.update_layout(
        xaxis_title="Current density (A/cm²)",
        yaxis_title="Peak wavelength (nm)"
    )


    # 4) f3dB vs current
    fig4 = px.line(
        df_graph,
        x="CurrentDensity",
        y="f3dB",
        title=f"{device} — f3dB vs Current density"
    )
    fig4.update_traces(mode="lines+markers")
    fig4.update_layout(
        xaxis_title="Current density (A/cm²)",
        yaxis_title="f-3dB (Hz)"
    )


    # 5) f3dB vs EQE
    fig5 = px.line(
        df_graph,
        x="EQE_interp",
        y="f3dB",
        title=f"{device} — f3dB vs EQE"
    )
    fig5.update_traces(mode="lines+markers")
    fig5.update_layout(
        xaxis_title="EQE (%)",
        yaxis_title="f-3dB (Hz)"
    )


    # 6) f3dB vs WPE
    fig6 = px.line(
        df_graph,
        x="WPE_interp",
        y="f3dB",
        title=f"{device} — f3dB vs WPE"
    )
    fig6.update_traces(mode="lines+markers")
    fig6.update_layout(
        xaxis_title="WPE (mW/W)",
        yaxis_title="f-3dB (Hz)"
    )

    # 7) f3dB vs L
    fig7 = px.line(
        df_graph,
        x="L_interp",
        y="f3dB",
        title=f"{device} — f-3dB vs L"
    )
    fig7.update_traces(mode="lines+markers")
    fig7.update_layout(
        xaxis_title="L (W)",
        yaxis_title="f-3dB (Hz)"
    )

    # 8) f3dB vs V
    fig8 = px.line(
        df_graph,
        x="V_interp",
        y="f3dB",
        title=f"{device} — f-3dB vs V"
    )
    fig8.update_traces(mode="lines+markers")
    fig8.update_layout(
        xaxis_title="Voltage (V)",
        yaxis_title="f-3dB (Hz)"
    )

    return fig1, fig2, fig3, fig4, fig5, fig6, fig7, fig8


if __name__ == "__main__":
    import warnings
    warnings.filterwarnings("ignore", category=UserWarning)
    app.run(debug=True, port=222)

In [6]:
liv_devices = set(df_liv2["Device"].unique())
freq_devices = set(df_final["Device"].unique())
common = liv_devices & freq_devices
print(df_final[df_final["Device"]=='67UKN4D6MMB7_3,1_210-D2'])
print("Commun :", common)
# only_freq = freq_devices - liv_devices
# print("Seulement dans f3dB :", only_freq)


        #    L  masked Ycoord Xcoord    I AbsI    V  Efficiency  Max  ...  \
2414  482  NaN     NaN    NaN    NaN  NaN  NaN  NaN         NaN  NaN  ...   
2415  483  NaN     NaN    NaN    NaN  NaN  NaN  NaN         NaN  NaN  ...   
2416  484  NaN     NaN    NaN    NaN  NaN  NaN  NaN         NaN  NaN  ...   
2417  485  NaN     NaN    NaN    NaN  NaN  NaN  NaN         NaN  NaN  ...   
2418  486  NaN     NaN    NaN    NaN  NaN  NaN  NaN         NaN  NaN  ...   
2419  487  NaN     NaN    NaN    NaN  NaN  NaN  NaN         NaN  NaN  ...   
2420  488  NaN     NaN    NaN    NaN  NaN  NaN  NaN         NaN  NaN  ...   
2421  489  NaN     NaN    NaN    NaN  NaN  NaN  NaN         NaN  NaN  ...   

              f3dB                                        frequencies  \
2414  2.090000e+08  10000000.0,19950000.0,29900000.0,39850000.0,49...   
2415  2.189500e+08  10000000.0,19950000.0,29900000.0,39850000.0,49...   
2416  2.488000e+08  10000000.0,19950000.0,29900000.0,39850000.0,49...   
2417  3.383500